# 1.Loading the dataSet

In [197]:
import pandas as pd
import numpy as np

data = pd.read_csv("/content/drive/MyDrive/Pandas_Data_Cleaning_Practice/datasets/independent_data_cleaning_practice.csv")
df =pd.DataFrame(data)
df

,row_no,order_id,customer_id,order_date,customer_name,city,product,category,quantity,unit_price,total_amount,payment_method,order_status,customer_email
0,1,ORD10244,CUST1029,2025-04-07,Aarav Sharma,Shivamogga,Mouse,Accessories,3,582.75,1748.25,Cash,Delivered,user62@example.com
1,2,ORD10409,CUST1021,2025-04-03,Sara Thomas,Mangaluru,Mouse,Accessories,2,614.19,1228.39,Credit Card,Cancelled,user163@example.com
2,3,ORD10297,CUST1158,2025.06.02,Diya Rao,Shivamogga,Keyboard,Accessories,4,1242.25,4969.0,Credit Card,Cancelled,user31@example.com
3,4,ORD10365,CUST1164,2025-08-12,Sara Thomas,Shivamogga,Keyboard,Accessories,2,1371.31,2742.62,Debit Card,Shipped,user182@example.com
4,5,ORD10205,CUST1128,2025-03-23,Sara Thomas,Hubballi,Webcam,Electronics,3,2267.38,6802.13,debit card,Returned,user6@example.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,423,ORD10202,CUST1156,2025.06.24,Ishaan Patel,Mangaluru,Mouse,Accessories,4,643.07,2572.26,Debit Card,Cancelled,user103@example.com
423,424,ORD10169,CUST1024,2025-08-12,Meera Nair,Hubballi,Mouse,Accessories,5,581.23,2906.16,Debit Card,Delivered,user107@example.com
424,425,ORD10186,CUST1008,2025-03-23,Ishaan Patel,Belagavi,Webcam,Electronics,4,2397.48,9589.93,NaN,Delivered,user33@example.com
425,426,ORD10036,CUST1130,2025-05-16,Sara Thomas,Mangaluru,Keyboard,Accessories,2,1267.5,2535.0,UPI,Complete,user163@example.com


# 2.Understanding the dataset

In [24]:
df.head()
df.info()
df.shape #427 rows and 14 columns before cleaning
df.columns
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 427 entries, 0 to 426
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   row_no          427 non-null    int64 
 1   order_id        427 non-null    object
 2   customer_id     427 non-null    object
 3   order_date      427 non-null    object
 4   customer_name   413 non-null    object
 5   city            411 non-null    object
 6   product         427 non-null    object
 7   category        427 non-null    object
 8   quantity        427 non-null    object
 9   unit_price      421 non-null    object
 10  total_amount    427 non-null    object
 11  payment_method  409 non-null    object
 12  order_status    427 non-null    object
 13  customer_email  405 non-null    object
dtypes: int64(1), object(13)
memory usage: 46.8+ KB


,row_no
count,427.000000
mean,214.000000
std,123.408536
min,1.000000
25%,107.500000
50%,214.000000
75%,320.500000
max,427.000000


# 3.Identifying The missing values

In [156]:
cdf = df.copy() # Deep copy of the data set to work on it
df.isnull().sum().sum() # Returns the total missing values
df.isnull().sum() # Returns the missing values in each column
#(df.isnull().sum())/len(df)*100  Returns the missing percentage of the value in the entire dataset

# customer_name 14 empty values
# city 16 empty values
# unite price 6 empty values
# payment method 18 empty values
# customer email 22 empty values
# total 76 missing values
cdf["total_amount"].isnull().sum() # 0 Missing values
cdf["unit_price"].isnull().sum() # 6 missing values



np.int64(6)

In [175]:
# Repacing the missing values with the unknown to respective columns

# first is the email - because we should not assume the someones email just like that so we will replace it with the Unknown

cdf["customer_email"].sample(10)
cdf["customer_email"].value_counts()
cdf["customer_email"].fillna("Unknown",inplace=True)
cdf["customer_email"].isnull().sum()
cdf["customer_email"].value_counts()
cdf["customer_email"].sample(10)

print("Unique Emails: ",cdf["customer_email"].nunique())
print("Total Emails: ",len(cdf["customer_email"]))
print("Missing emails: ",cdf["customer_email"].isnull().sum())



match_email = (
    r"^[A-Za-z0-9._%+-]+@"
    r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)
valid_emails = cdf["customer_email"].str.match(match_email,na=False)

invalid_emails =cdf[
    cdf["customer_email"].notna() & ~valid_emails
]
invalid_emails[
    ["customer_id", "customer_name", "customer_email"]
]
invalid_emails2 = invalid_emails

# print("Total Invalid Emails: ",len(invalid_emails)) 32 emails are invalid
for email in invalid_emails2["customer_email"]:
  cdf["customer_email"].replace(email,"unknown",inplace=True)

invalid_emails["customer_email"].value_counts()
cdf["customer_email"].sample(20)
cdf.loc[cdf["customer_email"].astype(str)=="Unknown"]
cdf["customer_email"].value_counts(dropna=False)





Unique Emails:  182
Total Emails:  427
Missing emails:  0


/tmp/ipykernel_593/1103326046.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cdf["customer_email"].fillna("Unknown",inplace=True)


,count
customer_email,
unknown,32
user24@example.com,8
user163@example.com,7
user62@example.com,6
user4@example.com,6
...,...
user78@example.com,1
user133@example.com,1
user93@example.com,1


In [173]:
# looking into the order status column
cdf["order_status"] = df["order_status"]
cdf["order_status"].value_counts()
cdf["order_status"] = (
    cdf["order_status"]
    .astype(str)
    .str.strip()
    .str.title()
    .replace("Canceled","Cancelled")
)
cdf["order_status"].value_counts(dropna=False)



,count
order_status,
Returned,130
Delivered,105
Cancelled,101
Shipped,90
Complete,1


In [174]:
# looking into the payment Method column

cdf["payment_method"].value_counts()
cdf["payment_method"] = (
  cdf["payment_method"]
  .astype(str)
  .str.strip()
  .str.title()
  .replace("Nan","Unknown")
)
cdf["payment_method"].value_counts(dropna=False)



,count
payment_method,
Net Banking,93
Cash,88
Credit Card,78
Upi,77
Debit Card,73
Unknown,18


In [203]:
# Looking for the total amount column
cdf["total_amount"].value_counts(dropna=False)
cdf["total_amount"] = pd.to_numeric(cdf["total_amount"],errors="coerce") #converted the datatype from object to float64

cdf.loc[(cdf["total_amount"]<0),"total_amount"]=np.nan
cdf["total_amount"].value_counts(dropna=False)
cdf["total_amount"].dtype

dtype('float64')

In [188]:
# Looking into the unit price column

cdf["unit_price"].isnull().sum()
cdf["unit_price"].value_counts(dropna=False)
cdf["unit_price"].dtypes
cdf["unit_price"] = pd.to_numeric(cdf["unit_price"],errors="coerce")
cdf["unit_price"].value_counts(dropna=False)
cdf.sample(20)


,row_no,order_id,customer_id,order_date,customer_name,city,product,category,quantity,unit_price,total_amount,payment_method,order_status,customer_email
282,283,ORD10199,CUST1145,22-06-2025,Aarav Sharma,Bengaluru,Webcam,Electronics,5,999999.00,13327.37,Upi,Delivered,user165@example.com
103,104,ORD10301,CUST1168,2025-09-16,Sara Thomas,Hubballi,Headphones,Accessories,4,2050.47,8201.87,Cash,Returned,user182@example.com
98,99,ORD10138,CUST1030,2025-07-08,NaN,Belagavi,Laptop,Electronics,3,60698.66,182095.98,Credit Card,Returned,user217@example.com
249,250,ORD10414,CUST1087,2025-09-26,Rohan Das,Shivamogga,Keyboard,Accessories,4,1282.02,5128.08,Credit Card,Cancelled,user126@example.com
26,27,ORD10004,CUST1092,2025-02-19,Ananya Iyer,Belagavi,Keyboard,Accessories,1,1053.39,9999999,Debit Card,Delivered,user187@example.com
24,25,ORD10045,CUST1112,2025-06-10,Diya Rao,Hubballi,Monitor,Electronics,-2,11444.47,45777.89,Net Banking,Shipped,unknown
23,24,ORD10206,CUST1056,2025-05-04,Ananya Iyer,Bengaluru,Laptop,Electronics,4,55089.83,220359.33,Upi,Cancelled,unknown
337,338,ORD10042,CUST1090,2025-08-24,Kabir Khan,Mysuru,Laptop,Electronics,4,50420.16,201680.63,Credit Card,Delivered,user169@example.com
380,381,ORD10262,CUST1005,2025-09-07,Aarav Sharma,Belagavi,Monitor,Electronics,5,13485.91,67429.53,Net Banking,Returned,user4@example.com
309,310,ORD10023,CUST1014,2025-08-28,Diya Rao,Bengaluru,Mouse,Accessories,3,635.10,1905.29,Cash,Cancelled,user205@example.com


In [15]:
from github_config import uploader
uploader.upload(
    commit_message="Successfully cleaned customer_email, order_status, payment_method, total_amount, unit_price columns"
)

Notebook uploaded successfully!
Commit message: Initial Change for clean up
Commit URL: https://github.com/faizankhan291106/pandas-data-cleaning-practice/commit/a1463d353df58366277078b7bd045d5200b52f38
